In [ ]:

import os
import json
import argparse
from pathlib import Path
from collections import deque

import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================
# CONFIGURATION
# ============================================

RAW_DIR = Path("dataset/raw")
PROCESSED_DIR = Path("dataset/processed")
MODEL_DIR = Path("models")

SEQUENCE_LENGTH = 30
NUM_HANDS = 2
FEATURES_PER_HAND = 63
FEATURES_PER_FRAME = 128

MODEL_PATH = MODEL_DIR / "isl_cnn_bilstm.keras"
LABEL_PATH = MODEL_DIR / "labels.json"
DATA_PATH = PROCESSED_DIR / "isl_features.npz"

VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv", ".mpeg"
}

MODEL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils


# ============================================
# 1. LANDMARK FEATURE EXTRACTION
# ============================================

def extract_landmarks(results):

    features = np.zeros(
        (NUM_HANDS, FEATURES_PER_HAND),
        dtype=np.float32
    )

    presence = np.zeros(
        NUM_HANDS,
        dtype=np.float32
    )

    if results.multi_hand_landmarks:

        for i, hand in enumerate(
            results.multi_hand_landmarks
        ):

            if not results.multi_handedness:
                continue

            label = (
                results.multi_handedness[i]
                .classification[0].label.lower()
            )

            # Fixed order: left, then right
            slot = 0 if label == "left" else 1

            points = np.array(
                [[lm.x, lm.y, lm.z] for lm in hand.landmark],
                dtype=np.float32
            )

            # Normalise relative to wrist
            points -= points[0]

            scale = np.max(np.abs(points[:, :2]))

            if scale > 0:
                points /= scale

            features[slot] = points.flatten()
            presence[slot] = 1.0

    return np.concatenate(
        [features.flatten(), presence]
    ).astype(np.float32)


# ============================================
# 2. PROCESS ONE VIDEO
# ============================================

def process_video(video_path, hands):

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print("Cannot open:", video_path)
        return None

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if total_frames < 2:
        cap.release()
        return None

    # Select 30 frames distributed across the video
    frame_indices = np.linspace(
        0,
        total_frames - 1,
        SEQUENCE_LENGTH
    ).astype(int)

    sequence = []

    for frame_index in frame_indices:

        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))

        success, frame = cap.read()

        if not success:
            sequence.append(
                np.zeros(FEATURES_PER_FRAME, dtype=np.float32)
            )
            continue

        # Mirror before MediaPipe detection
        frame = cv2.flip(frame, 1)

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        results = hands.process(rgb)

        features = extract_landmarks(results)
        sequence.append(features)

    cap.release()

    sequence = np.array(sequence, dtype=np.float32)

    if sequence.shape != (
        SEQUENCE_LENGTH,
        FEATURES_PER_FRAME
    ):
        return None

    # Reject videos where no hands were detected
    if np.sum(sequence[:, -2:]) == 0:
        return None

    return sequence


# ============================================
# 3. DATASET PREPARATION
# ============================================

def prepare_dataset():

    if not RAW_DIR.exists():
        raise FileNotFoundError(
            "Create dataset/raw and add labelled video folders."
        )

    X = []
    y = []

    class_folders = sorted([
        folder for folder in RAW_DIR.iterdir()
        if folder.is_dir()
    ])

    if not class_folders:
        raise ValueError(
            "No sign folders found inside dataset/raw."
        )

    print("Starting video preprocessing...")

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as hands:

        for class_folder in class_folders:

            label = class_folder.name

            videos = [
                path for path in class_folder.rglob("*")
                if path.is_file()
                and path.suffix.lower() in VIDEO_EXTENSIONS
            ]

            print(
                f"\nProcessing {label}: {len(videos)} videos"
            )

            for video_path in videos:

                sequence = process_video(
                    video_path,
                    hands
                )

                if sequence is None:
                    print("Skipped:", video_path.name)
                    continue

                X.append(sequence)
                y.append(label)

                print("Processed:", video_path.name)

    if not X:
        raise ValueError(
            "No usable videos found. Check the dataset folders."
        )

    X = np.array(X, dtype=np.float32)
    y = np.array(y)

    np.savez_compressed(
        DATA_PATH,
        X=X,
        y=y
    )

    print("\nDataset preprocessing completed.")
    print("Total videos processed:", len(X))
    print("Number of classes:", len(set(y)))
    print("Saved:", DATA_PATH)


# ============================================
# 4. LOAD PROCESSED DATA
# ============================================

def load_dataset():

    if not DATA_PATH.exists():
        raise FileNotFoundError(
            "Run prepare mode before training."
        )

    data = np.load(DATA_PATH)

    X = data["X"]
    y = data["y"]

    encoder = LabelEncoder()
    y = encoder.fit_transform(y)

    return X, y, encoder


# ============================================
# 5. CNN-BILSTM MODEL
# ============================================

def build_model(num_classes):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(
                SEQUENCE_LENGTH,
                FEATURES_PER_FRAME
            )
        ),

        tf.keras.layers.Conv1D(
            64,
            kernel_size=3,
            activation="relu"
        ),

        tf.keras.layers.MaxPooling1D(
            pool_size=2
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(
                64,
                return_sequences=True
            )
        ),

        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(
            num_classes,
            activation="softmax"
        )
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# ============================================
# 6. MODEL EVALUATION
# ============================================

def evaluate_model(model, X_test, y_test, classes):

    probabilities = model.predict(
        X_test,
        verbose=0
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    print("\nModel accuracy:")
    print(f"{accuracy * 100:.2f}%")

    print("\nClassification report:")

    print(
        classification_report(
            y_test,
            predictions,
            labels=np.arange(len(classes)),
            target_names=classes,
            zero_division=0
        )
    )

    matrix = confusion_matrix(
        y_test,
        predictions,
        labels=np.arange(len(classes))
    )

    plt.figure(figsize=(12, 10))

    plt.imshow(matrix, cmap="Blues")

    plt.title("ISL Recognition Confusion Matrix")
    plt.xlabel("Predicted Sign")
    plt.ylabel("Actual Sign")

    plt.xticks(
        np.arange(len(classes)),
        classes,
        rotation=90
    )

    plt.yticks(
        np.arange(len(classes)),
        classes
    )

    plt.colorbar()
    plt.tight_layout()

    plt.savefig(
        MODEL_DIR / "confusion_matrix.png"
    )

    plt.show()


# ============================================
# 7. MODEL TRAINING
# ============================================

def train_model():

    print("Loading processed dataset...")

    X, y, encoder = load_dataset()

    classes, counts = np.unique(
        y,
        return_counts=True
    )

    if len(classes) < 2:
        raise ValueError(
            "At least two sign classes are required."
        )

    if np.min(counts) < 5:
        raise ValueError(
            "Each class needs at least five usable videos."
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    print("Training samples:", len(X_train))
    print("Testing samples:", len(X_test))
    print("Classes:", len(encoder.classes_))

    model = build_model(
        len(encoder.classes_)
    )

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    print("\nTraining CNN-BiLSTM model...")

    model.fit(
        X_train,
        y_train,
        epochs=50,
        batch_size=8,
        validation_split=0.20,
        callbacks=[early_stopping],
        shuffle=True
    )

    model.save(MODEL_PATH)

    with open(LABEL_PATH, "w") as file:
        json.dump(
            encoder.classes_.tolist(),
            file
        )

    print("\nModel saved successfully.")
    print("Model:", MODEL_PATH)

    evaluate_model(
        model,
        X_test,
        y_test,
        encoder.classes_
    )


# ============================================
# 8. REAL-TIME WEBCAM RECOGNITION
# ============================================

def predict_sign():

    if not MODEL_PATH.exists():
        print("Train the model first.")
        return

    model = tf.keras.models.load_model(
        MODEL_PATH
    )

    with open(LABEL_PATH, "r") as file:
        classes = json.load(file)

    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Webcam unavailable.")
        return

    sequence = deque(
        maxlen=SEQUENCE_LENGTH
    )

    recent_predictions = deque(maxlen=5)

    current_label = "Waiting..."
    confidence = 0.0

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as hands:

        while True:

            success, frame = cap.read()

            if not success:
                break

            frame = cv2.flip(frame, 1)

            rgb = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            results = hands.process(rgb)

            if results.multi_hand_landmarks:

                for hand in results.multi_hand_landmarks:

                    mp_drawing.draw_landmarks(
                        frame,
                        hand,
                        mp_hands.HAND_CONNECTIONS
                    )

            features = extract_landmarks(results)

            sequence.append(features)

            if len(sequence) == SEQUENCE_LENGTH:

                input_data = np.expand_dims(
                    np.array(sequence),
                    axis=0
                )

                probabilities = model.predict(
                    input_data,
                    verbose=0
                )[0]

                index = int(
                    np.argmax(probabilities)
                )

                confidence = float(
                    probabilities[index]
                )

                if confidence >= 0.70:

                    recent_predictions.append(index)

                    # Most frequent recent prediction
                    current_index = max(
                        set(recent_predictions),
                        key=recent_predictions.count
                    )

                    current_label = classes[current_index]

                else:
                    current_label = "Uncertain"

            cv2.rectangle(
                frame,
                (0, 0),
                (frame.shape[1], 100),
                (30, 30, 30),
                -1
            )

            cv2.putText(
                frame,
                f"ISL Sign: {current_label}",
                (15, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                f"Confidence: {confidence * 100:.1f}%",
                (15, 78),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2
            )

            cv2.imshow(
                "Indian Sign Language Recognition",
                frame
            )

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()


# ============================================
# 9. MAIN PROGRAM
# ============================================

def main():

    parser = argparse.ArgumentParser(
        description="Indian Sign Language Recognition"
    )

    parser.add_argument(
        "mode",
        choices=["prepare", "train", "predict"],
        help="Select project operation"
    )

    args = parser.parse_args()

    if args.mode == "prepare":
        prepare_dataset()

    elif args.mode == "train":
        train_model()

    elif args.mode == "predict":
        predict_sign()


if __name__ == "__main__":
    main()
